# Milestone 2 – Klasteryzacja sygnałów z sensorów

**Aktywności = Ground Truth.** Cel: zbadać, czy na podstawie samych sygnałów
sensorycznych można odtworzyć podział na aktywności – i która metoda klasteryzacji
robi to najlepiej.

Pipeline: NaN/outliery → normalizacja → (PCA) → klasteryzacja (K-Means, DBSCAN, HDBSCAN)
→ porównanie z GT → wybór najlepszej metody → macierz przejść


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
SEED = 42
np.random.seed(SEED)

# Wczytanie danych
candidate_dirs = [Path('.'), Path('Dataset_8087219'), Path('..') / 'Dataset_8087219']
DATA_DIR = None
for d in candidate_dirs:
    if (d / 'Signature_Burn.txt').exists():
        DATA_DIR = d.resolve()
        break
if DATA_DIR is None:
    raise FileNotFoundError('Nie znaleziono folderu z plikami Signature_*.txt.')

signature_files = sorted(DATA_DIR.glob('Signature_*.txt'))
records = []
for fp in signature_files:
    activity = fp.stem.replace('Signature_', '')
    with fp.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            obj = json.loads(line)
            row = {'activity': activity, 'event_index': idx,
                   'station': obj.get('station'), 'timestamp_raw': obj.get('timestamp')}
            for k, v in obj.get('events', {}).items():
                row[k] = v
            records.append(row)

df = pd.DataFrame(records)
df['timestamp'] = pd.to_datetime(df['timestamp_raw'], utc=True, errors='coerce')
META = ['activity','event_index','station','timestamp_raw','timestamp']
SIG_COLS = [c for c in df.columns if c not in META]
activities = sorted(df['activity'].unique())
y_true = df['activity'].values  # Ground Truth

print(f'Eventow: {len(df)}, Sygnalow: {len(SIG_COLS)}, Aktywnosci (GT): {len(activities)}')
print(f'Aktywnosci: {activities}')


## 1. Analiza NaN i outlierów w sygnałach


In [ ]:
# Analiza NaN
nan_per_col = df[SIG_COLS].isna().sum()
print('=== BRAKUJACE WARTOSCI (NaN) PER SYGNAL ===')
for col in SIG_COLS:
    n = nan_per_col[col]
    active_for = ', '.join(df[df[col].notna()]['activity'].unique())
    print(f'  {col:22s}: {n:3d} NaN ({n/len(df)*100:4.0f}%) | aktywny dla: {active_for}')

print(f'\nStrategia: NaN -> 0 (sygnal nieaktywny = komponent nie pracuje)')

# Outliery w wartosciach sygnalow
all_vals = df[SIG_COLS].stack().dropna().unique()
print(f'\nWartosci unikalne PRZED normalizacja: {sorted(all_vals)}')
print('Brak outlierow numerycznych - dane dyskretne {-512, 0, 1, 512}')

# Outliery czasowe
df_s = df.sort_values(['activity','event_index']).copy()
df_s['dt'] = df_s.groupby('activity')['timestamp'].diff().dt.total_seconds()
outliers_t = df_s[df_s['dt'].notna() & ((df_s['dt'] > 3) | (df_s['dt'] < 1))]
print(f'\nOutliery czasowe (odstep >3s lub <1s): {len(outliers_t)} z {df_s["dt"].notna().sum()}')
print('Interpretacja: artefakty rejestracji PLC, nie bledy procesu.')


## 2. Normalizacja sygnałów


In [ ]:
# Normalizacja: 512->1, -512->-1, NaN->0
df_norm = df.copy()
for col in SIG_COLS:
    df_norm[col] = df_norm[col].apply(lambda x: x/512 if pd.notna(x) and abs(x)>1 else x)
df_norm[SIG_COLS] = df_norm[SIG_COLS].fillna(0)

X = df_norm[SIG_COLS].values.astype(float)
print(f'Macierz cech: {X.shape}')
print(f'Wartosci unikalne PO normalizacji: {sorted(np.unique(X))}')

# Dodatkowe warianty normalizacji dla klasteryzacji
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

X_mm = MinMaxScaler().fit_transform(X)  # [0,1]
X_std = StandardScaler().fit_transform(X)  # z-score

# PCA
pca5 = PCA(n_components=5, random_state=SEED)
X_pca5 = pca5.fit_transform(X_mm)
pca2 = PCA(n_components=2, random_state=SEED)
X_pca2 = pca2.fit_transform(X_mm)

print(f'PCA 5 komponentow - wariancja: {pca5.explained_variance_ratio_.sum()*100:.1f}%')
print(f'PCA 2 komponenty  - wariancja: {pca2.explained_variance_ratio_.sum()*100:.1f}%')

# Wizualizacja PCA 2D
fig, ax = plt.subplots(figsize=(10, 7))
colors_act = plt.cm.Set1.colors
cmap = {a: colors_act[i] for i, a in enumerate(activities)}
for act in activities:
    mask = df['activity'] == act
    ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[cmap[act]], label=act,
              s=60, alpha=0.85, edgecolors='white', linewidths=0.5)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)
ax.set_title('PCA 2D - aktywnosci (Ground Truth)', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('m2_pca2d_gt.png', dpi=120, bbox_inches='tight')
plt.show()


## 3. Klasteryzacja – K-Means


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score

# Grid search K-Means
km_results = []
datasets = {'Raw_26D': X_mm, 'PCA_5': X_pca5, 'PCA_2': X_pca2}

for dname, Xd in datasets.items():
    for k in range(2, 10):
        km = KMeans(n_clusters=k, random_state=SEED, n_init=20)
        labels = km.fit_predict(Xd)
        ari = adjusted_rand_score(y_true, labels)
        nmi = normalized_mutual_info_score(y_true, labels)
        sil = silhouette_score(Xd, labels) if len(set(labels)) > 1 else -1
        km_results.append({'Dane': dname, 'k': k, 'ARI': round(ari, 3),
                           'NMI': round(nmi, 3), 'Silhouette': round(sil, 3)})

km_df = pd.DataFrame(km_results)
print('=== K-MEANS: GRID SEARCH ===')
print(km_df.to_string(index=False))

# Najlepszy wynik
best_km = km_df.loc[km_df['ARI'].idxmax()]
print(f'\nNajlepszy K-Means: {best_km["Dane"]}, k={best_km["k"]}, '
      f'ARI={best_km["ARI"]}, NMI={best_km["NMI"]}, Sil={best_km["Silhouette"]}')


In [ ]:
# Wizualizacja K-Means k=6 vs GT
best_data_name = best_km['Dane']
best_X = datasets[best_data_name]
km_best = KMeans(n_clusters=int(best_km['k']), random_state=SEED, n_init=20)
km_labels = km_best.fit_predict(best_X)

# Macierz konfuzji
print('=== MACIERZ KONFUZJI: K-Means vs Ground Truth ===')
ct = pd.crosstab(df['activity'], km_labels, rownames=['Aktywnosc (GT)'], colnames=['Klaster'])
print(ct.to_string())

# Wizualizacja na PCA 2D
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
cluster_colors = plt.cm.tab10.colors
for c in range(int(best_km['k'])):
    mask = km_labels == c
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[cluster_colors[c]],
                    label=f'Klaster {c}', s=60, alpha=0.85, edgecolors='white')
axes[0].set_title(f'K-Means k={int(best_km["k"])} ({best_data_name})\n'
                  f'ARI={best_km["ARI"]}, NMI={best_km["NMI"]}', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

for act in activities:
    mask = df['activity'] == act
    axes[1].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[cmap[act]],
                    label=act, s=60, alpha=0.85, edgecolors='white')
axes[1].set_title('Ground Truth (aktywnosci)', fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.tight_layout()
plt.savefig('m2_kmeans_vs_gt.png', dpi=120, bbox_inches='tight')
plt.show()


## 4. Klasteryzacja – DBSCAN


In [ ]:
from sklearn.cluster import DBSCAN

# Grid search DBSCAN
db_results = []
for eps in [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 2.5, 3.0]:
    for ms in [2, 3, 5]:
        db = DBSCAN(eps=eps, min_samples=ms)
        labels = db.fit_predict(X_pca5)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        noise_pct = n_noise / len(labels) * 100
        if n_clusters >= 2:
            mask_valid = labels != -1
            ari = adjusted_rand_score(y_true[mask_valid], labels[mask_valid]) if mask_valid.sum() > 1 else -1
            nmi = normalized_mutual_info_score(y_true[mask_valid], labels[mask_valid]) if mask_valid.sum() > 1 else -1
            sil = silhouette_score(X_pca5[mask_valid], labels[mask_valid]) if mask_valid.sum() > 1 and len(set(labels[mask_valid])) > 1 else -1
        else:
            ari, nmi, sil = -1, -1, -1
        db_results.append({'eps': eps, 'min_samples': ms, 'Klastry': n_clusters,
                           'Szum_%': round(noise_pct, 1), 'ARI': round(ari, 3),
                           'NMI': round(nmi, 3), 'Silhouette': round(sil, 3)})

db_df = pd.DataFrame(db_results)
print('=== DBSCAN: GRID SEARCH (dane: PCA-5, StandardScaler) ===')
print(db_df[db_df['Klastry'] >= 2].to_string(index=False))

valid_db = db_df[db_df['ARI'] > 0]
if len(valid_db) > 0:
    best_db = valid_db.loc[valid_db['ARI'].idxmax()]
    print(f'\nNajlepszy DBSCAN: eps={best_db["eps"]}, min_samples={best_db["min_samples"]}, '
          f'Klastry={best_db["Klastry"]}, ARI={best_db["ARI"]}, Szum={best_db["Szum_%"]}%')
else:
    print('\nDBSCAN nie znalazl sensownych klastrow dla tych parametrow.')
    best_db = db_df.iloc[0]  # placeholder


## 5. Klasteryzacja – HDBSCAN


In [ ]:
import hdbscan

# Grid search HDBSCAN
hdb_results = []
for mcs in [3, 5, 7, 10, 15]:
    for ms in [2, 3, 5]:
        hdb = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=ms)
        labels = hdb.fit_predict(X_pca5)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = (labels == -1).sum()
        noise_pct = n_noise / len(labels) * 100
        if n_clusters >= 2:
            mask_valid = labels != -1
            ari = adjusted_rand_score(y_true[mask_valid], labels[mask_valid]) if mask_valid.sum() > 1 else -1
            nmi = normalized_mutual_info_score(y_true[mask_valid], labels[mask_valid]) if mask_valid.sum() > 1 else -1
            sil = silhouette_score(X_pca5[mask_valid], labels[mask_valid]) if mask_valid.sum() > 1 and len(set(labels[mask_valid])) > 1 else -1
        else:
            ari, nmi, sil = -1, -1, -1
        hdb_results.append({'min_cluster_size': mcs, 'min_samples': ms,
                            'Klastry': n_clusters, 'Szum_%': round(noise_pct, 1),
                            'ARI': round(ari, 3), 'NMI': round(nmi, 3),
                            'Silhouette': round(sil, 3)})

hdb_df = pd.DataFrame(hdb_results)
print('=== HDBSCAN: GRID SEARCH (dane: PCA-5) ===')
print(hdb_df[hdb_df['Klastry'] >= 2].to_string(index=False))

valid_hdb = hdb_df[hdb_df['ARI'] > 0]
if len(valid_hdb) > 0:
    best_hdb = valid_hdb.loc[valid_hdb['ARI'].idxmax()]
    print(f'\nNajlepszy HDBSCAN: mcs={best_hdb["min_cluster_size"]}, ms={best_hdb["min_samples"]}, '
          f'Klastry={best_hdb["Klastry"]}, ARI={best_hdb["ARI"]}, Szum={best_hdb["Szum_%"]}%')
else:
    print('\nHDBSCAN nie znalazl sensownych klastrow.')
    best_hdb = hdb_df.iloc[0]


## 6. Porównanie klastrów z aktywnościami (Ground Truth)


In [ ]:
# Porownanie zbiorcze
print('=' * 70)
print('POROWNANIE ZBIORCZE ALGORYTMOW KLASTERYZACJI')
print('=' * 70)

comparison = []
comparison.append({'Algorytm': 'K-Means', 'Parametry': f'k={int(best_km["k"])}, dane={best_km["Dane"]}',
                   'ARI': best_km['ARI'], 'NMI': best_km['NMI'],
                   'Silhouette': best_km['Silhouette'], 'Klastry': int(best_km['k']), 'Szum_%': 0})

if len(valid_db) > 0:
    comparison.append({'Algorytm': 'DBSCAN',
                       'Parametry': f'eps={best_db["eps"]}, ms={best_db["min_samples"]}',
                       'ARI': best_db['ARI'], 'NMI': best_db['NMI'],
                       'Silhouette': best_db['Silhouette'],
                       'Klastry': int(best_db['Klastry']), 'Szum_%': best_db['Szum_%']})

if len(valid_hdb) > 0:
    comparison.append({'Algorytm': 'HDBSCAN',
                       'Parametry': f'mcs={int(best_hdb["min_cluster_size"])}, ms={int(best_hdb["min_samples"])}',
                       'ARI': best_hdb['ARI'], 'NMI': best_hdb['NMI'],
                       'Silhouette': best_hdb['Silhouette'],
                       'Klastry': int(best_hdb['Klastry']), 'Szum_%': best_hdb['Szum_%']})

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False))

# Wybor najlepszego
best_overall = comp_df.loc[comp_df['ARI'].idxmax()]
print(f'\n>>> NAJLEPSZA METODA: {best_overall["Algorytm"]} ({best_overall["Parametry"]})')
print(f'    ARI={best_overall["ARI"]}, NMI={best_overall["NMI"]}, Silhouette={best_overall["Silhouette"]}')


In [ ]:
# Macierz konfuzji dla kazdego algorytmu
print('=== MACIERZE KONFUZJI: KLASTRY vs AKTYWNOSCI (GT) ===\n')

# K-Means (najlepszy)
print(f'--- K-Means (k={int(best_km["k"])}, {best_km["Dane"]}) ---')
km_final = KMeans(n_clusters=int(best_km['k']), random_state=SEED, n_init=20)
km_final_labels = km_final.fit_predict(datasets[best_km['Dane']])
ct_km = pd.crosstab(df['activity'], km_final_labels, rownames=['Aktywnosc'], colnames=['Klaster'])
print(ct_km.to_string())
print()

# DBSCAN (najlepszy)
if len(valid_db) > 0:
    print(f'--- DBSCAN (eps={best_db["eps"]}, min_samples={best_db["min_samples"]}) ---')
    db_final = DBSCAN(eps=float(best_db['eps']), min_samples=int(best_db['min_samples']))
    db_final_labels = db_final.fit_predict(X_pca5)
    ct_db = pd.crosstab(df['activity'], db_final_labels, rownames=['Aktywnosc'], colnames=['Klaster'])
    print(ct_db.to_string())
    print(f'  Szum (klaster -1): {(db_final_labels == -1).sum()} eventow')
    print()

# HDBSCAN (najlepszy)
if len(valid_hdb) > 0:
    print(f'--- HDBSCAN (mcs={int(best_hdb["min_cluster_size"])}, ms={int(best_hdb["min_samples"])}) ---')
    hdb_final = hdbscan.HDBSCAN(min_cluster_size=int(best_hdb['min_cluster_size']),
                                 min_samples=int(best_hdb['min_samples']))
    hdb_final_labels = hdb_final.fit_predict(X_pca5)
    ct_hdb = pd.crosstab(df['activity'], hdb_final_labels, rownames=['Aktywnosc'], colnames=['Klaster'])
    print(ct_hdb.to_string())
    print(f'  Szum (klaster -1): {(hdb_final_labels == -1).sum()} eventow')


## 7. Macierz przejść między klastrami

Dla najlepszej metody klasteryzacji obliczamy macierz przejść:
ile razy event z klastra A jest bezpośrednio następowany przez event z klastra B
(w porządku chronologicznym wewnątrz każdej aktywności).


In [ ]:
# Macierz przejsc dla NAJLEPSZEJ metody (DBSCAN/HDBSCAN)
# Wybieramy algorytm z najwyzszym ARI
best_overall = comp_df.loc[comp_df['ARI'].idxmax()]
best_algo = best_overall['Algorytm']

if best_algo == 'DBSCAN' and len(valid_db) > 0:
    best_labels = db_final_labels
    best_algo_name = f'DBSCAN (eps={best_db["eps"]}, ms={int(best_db["min_samples"])})'
elif best_algo == 'HDBSCAN' and len(valid_hdb) > 0:
    best_labels = hdb_final_labels
    best_algo_name = f'HDBSCAN (mcs={int(best_hdb["min_cluster_size"])}, ms={int(best_hdb["min_samples"])})'
else:
    best_labels = km_final_labels
    best_algo_name = f'K-Means k={int(best_km["k"])}'

print(f'Najlepsza metoda: {best_algo_name}')
print(f'ARI={best_overall["ARI"]}, NMI={best_overall["NMI"]}')

# Macierz przejsc (uwzgledniamy tylko klastry, nie szum -1)
unique_labels = sorted(set(best_labels))
n_clusters_best = len(unique_labels)
T = np.zeros((n_clusters_best, n_clusters_best), dtype=int)
lbl_idx = {l: i for i, l in enumerate(unique_labels)}

for act in activities:
    mask = df['activity'] == act
    act_labels = best_labels[mask]
    for i in range(len(act_labels) - 1):
        T[lbl_idx[act_labels[i]], lbl_idx[act_labels[i+1]]] += 1

labels_str = [f'K{l}' if l != -1 else 'Szum' for l in unique_labels]
print(f'\n=== MACIERZ PRZEJSC MIEDZY KLASTRAMI ({best_algo_name}) ===')
T_df = pd.DataFrame(T, index=labels_str, columns=labels_str)
print(T_df.to_string())

# Wizualizacja
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(T, cmap='Blues', aspect='auto')
plt.colorbar(im, ax=ax, shrink=0.8, label='Liczba przejsc')
ax.set_xticks(range(n_clusters_best))
ax.set_yticks(range(n_clusters_best))
ax.set_xticklabels(labels_str, fontsize=10)
ax.set_yticklabels(labels_str, fontsize=10)
ax.set_xlabel('Klaster docelowy')
ax.set_ylabel('Klaster zrodlowy')
ax.set_title(f'Macierz przejsc miedzy klastrami\n{best_algo_name}', fontweight='bold')
for i in range(n_clusters_best):
    for j in range(n_clusters_best):
        if T[i, j] > 0:
            ax.text(j, i, str(T[i, j]), ha='center', va='center', fontsize=11,
                    color='white' if T[i, j] > T.max()*0.5 else 'black', fontweight='bold')
plt.tight_layout()
plt.savefig('m2_transition_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_transition_matrix.png')

# Interpretacja
diag_sum = np.trace(T)
total = T.sum()
print(f'\n=== INTERPRETACJA ===')
print(f'Przejscia na diagonali: {diag_sum}/{total} ({diag_sum/total*100:.1f}%)')
print(f'Przejscia miedzy klastrami: {total-diag_sum}/{total} ({(total-diag_sum)/total*100:.1f}%)')

print('\n=== MAPOWANIE KLASTROW NA AKTYWNOSCI ===')
for c in unique_labels:
    mask = best_labels == c
    acts_in_cluster = df['activity'][mask].value_counts()
    label_name = f'K{c}' if c != -1 else 'Szum'
    print(f'  {label_name}: {dict(acts_in_cluster)} (n={mask.sum()})')

# Dodatkowy wykres: klastry vs GT na PCA 2D
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for c in unique_labels:
    mask = best_labels == c
    label_name = f'K{c}' if c != -1 else 'Szum'
    color = cluster_colors[c % 10] if c != -1 else 'lightgray'
    axes[0].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[color], label=label_name,
                    s=60, alpha=0.85, edgecolors='white')
axes[0].set_title(f'Klastry: {best_algo_name}\nARI={best_overall["ARI"]}, NMI={best_overall["NMI"]}',
                  fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

for act in activities:
    mask = df['activity'] == act
    axes[1].scatter(X_pca2[mask, 0], X_pca2[mask, 1], c=[cmap[act]], label=act,
                    s=60, alpha=0.85, edgecolors='white')
axes[1].set_title('Ground Truth (aktywnosci)', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
plt.tight_layout()
plt.savefig('m2_dbscan_vs_gt.png', dpi=120, bbox_inches='tight')
plt.show()
print('Wykres zapisany: m2_dbscan_vs_gt.png')


## 8. Podsumowanie

### Pipeline:
1. Sygnały z czujników (119 eventów × 26 sygnałów)
2. NaN → 0, normalizacja 512→1
3. Analiza outlierów (brak krytycznych)
4. PCA (5 komponentów, ~72.5% wariancji)
5. Klasteryzacja: K-Means, DBSCAN, HDBSCAN (grid search)
6. Porównanie z Ground Truth: ARI, NMI, macierz konfuzji
7. Wybór najlepszej metody (DBSCAN ARI=0.914) → macierz przejść między klastrami

### Wyniki:
- **K-Means** (k=4, Raw_26D): ARI=0.790, NMI=0.867
- **DBSCAN** (eps=1.0, ms=2, PCA-5): **ARI=0.914**, NMI=0.944 — **najlepszy**
- **HDBSCAN** (mcs=10, ms=5, PCA-5): ARI=0.914, NMI=0.944

### Wnioski:
- DBSCAN/HDBSCAN znajdują 5 klastrów (zamiast 6) — łączą Burn+Transport
- 4 z 6 aktywności mają czyste klastry (Mill, Pickup, Sort, Storage)
- Wszystkie przejścia na diagonali — klastry = całe aktywności
